# Otto Group Product Classification — Data Understanding

## 1. Data Loading

**Objective:** Load the training, test, and sample submission datasets and verify their basic dimensions before conducting exploratory data analysis.

**Methodology:** Read the original CSV files using Pandas and inspect the number of rows and columns in each dataset.

In [1]:
from pathlib import Path

import pandas as pd

current_dir = Path.cwd()
project_root = (
    current_dir.parent
    if current_dir.name == "notebooks"
    else current_dir
)

data_dir = project_root / "data" / "raw"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sampleSubmission.csv")

print("Working directory:", current_dir)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Working directory: /Users/ebrietas/otto-group-product-classification/notebooks
Train shape: (61878, 95)
Test shape: (144368, 94)
Sample submission shape: (144368, 10)


In [2]:
print("First train columns:", train.columns[:5].tolist())
print("Last train columns:", train.columns[-5:].tolist())

train_only = sorted(set(train.columns) - set(test.columns))
test_only = sorted(set(test.columns) - set(train.columns))

print("\nTrain-only columns:", train_only)
print("Test-only columns:", test_only)

print("\nTarget classes:", sorted(train["target"].unique()))
print("Submission columns:", sample_submission.columns.tolist())

First train columns: ['id', 'feat_1', 'feat_2', 'feat_3', 'feat_4']
Last train columns: ['feat_90', 'feat_91', 'feat_92', 'feat_93', 'target']

Train-only columns: ['target']
Test-only columns: []

Target classes: ['Class_1', 'Class_2', 'Class_3', 'Class_4', 'Class_5', 'Class_6', 'Class_7', 'Class_8', 'Class_9']
Submission columns: ['id', 'Class_1', 'Class_2', 'Class_3', 'Class_4', 'Class_5', 'Class_6', 'Class_7', 'Class_8', 'Class_9']


### Schema Validation

**Objective:** Verify the consistency of the training, test, and submission schemas and identify the target classes.

**Methodology:** Compare column names across datasets, identify training-only columns, and inspect the unique target labels and submission format.

**Findings:**
- The training dataset contains 61,878 observations and 95 columns.
- The test dataset contains 144,368 observations and 94 columns.
- Both datasets share 93 predictive features (`feat_1`–`feat_93`) and an `id` column.
- The `target` column appears only in the training dataset.
- The target contains nine categories (`Class_1`–`Class_9`).
- The sample submission contains an identifier and nine class-probability columns.

**Modeling Decision:** Exclude `id` and `target` from the predictive features. Use the 93 shared feature columns for modeling and preserve the class ordering required by the submission format.

In [3]:
class_distribution = (
    train["target"]
    .value_counts()
    .sort_index()
    .rename_axis("class")
    .to_frame("count")
)

class_distribution["percentage"] = (
    class_distribution["count"] / len(train) * 100
).round(2)

display(class_distribution)

,count,percentage
class,,
Class_1,1929,3.12
Class_2,16122,26.05
Class_3,8004,12.94
Class_4,2691,4.35
Class_5,2739,4.43
Class_6,14135,22.84
Class_7,2839,4.59
Class_8,8464,13.68
Class_9,4955,8.01


### Target Class Distribution

**Objective:** Examine the distribution of the nine target classes and identify potential class imbalance.

**Methodology:** Calculate the frequency and percentage of observations belonging to each target class.

**Findings:**
- The dataset contains nine target classes with unequal frequencies.
- `Class_2` is the most frequent class, representing 26.05% of observations, followed by `Class_6` at 22.84%.
- `Class_1` is the least frequent class, representing 3.12% of observations.
- The most frequent class contains approximately 8.4 times as many observations as the least frequent class.

**Interpretation:** The dataset exhibits substantial class imbalance. This may affect predictive performance across individual classes, particularly those with fewer training examples. However, class-specific performance must be evaluated before drawing conclusions.

**Modeling Decision:** Use stratified data splitting to preserve class proportions across training and validation sets. Evaluate models using multiclass log loss, consistent with the competition metric, and consider class-level diagnostics when assessing model performance.

In [4]:
feature_columns = [f"feat_{i}" for i in range(1, 94)]

print("Training feature types:")
print(train[feature_columns].dtypes.value_counts())

print("\nTest feature types:")
print(test[feature_columns].dtypes.value_counts())

print("\nMissing values in train features:")
print(train[feature_columns].isna().sum().sum())

print("\nMissing values in test features:")
print(test[feature_columns].isna().sum().sum())

print("\nMissing target values:")
print(train["target"].isna().sum())

Training feature types:
int64    93
Name: count, dtype: int64

Test feature types:
int64    93
Name: count, dtype: int64

Missing values in train features:
0

Missing values in test features:
0

Missing target values:
0


### Feature Data Types and Missing Values

**Objective:** Identify missing values and inspect feature data types to determine the initial preprocessing requirements.

**Methodology:** Examine the data types of all 93 predictive features and count missing values in the training features, test features, and target variable.

**Findings:**
- All 93 predictive features are stored as `int64` in both datasets.
- No missing values were detected in the training or test features.
- The target variable contains no missing values.

**Interpretation:** The datasets are complete with respect to missing values, and the predictive features are represented numerically.

**Modeling Decision:** No missing-value imputation or categorical feature encoding is required at this stage. Feature scaling will be considered according to the requirements of the selected modeling algorithm.

In [5]:
X = train[feature_columns]

constant_features = X.columns[X.nunique() == 1].tolist()
zero_percentage = X.eq(0).mean() * 100

print("Constant features:", constant_features)
print("Minimum feature value:", X.min().min())
print("Maximum feature value:", X.max().max())
print("Median zero percentage:", round(zero_percentage.median(), 2))

print("\nFive features with the highest zero percentages:")
print(zero_percentage.sort_values(ascending=False).head().round(2))

Constant features: []
Minimum feature value: 0
Maximum feature value: 352
Median zero percentage: 83.2

Five features with the highest zero percentages:
feat_6     98.11
feat_84    97.70
feat_51    97.22
feat_5     95.20
feat_81    94.86
dtype: float64


### Constant Features and Zero-Value Analysis

**Objective:** Identify constant features and examine the prevalence of zero values across the predictive features.

**Methodology:** Calculate the number of unique values per feature, inspect the overall value range, and measure the percentage of zero values in each feature.

**Findings:**
- No constant features were detected.
- Feature values range from 0 to 352 in the training dataset.
- The median zero percentage across the 93 features is 83.20%.
- `feat_6` has the highest observed zero percentage at 98.11%, followed by `feat_84` at 97.70%.

**Interpretation:** The features exhibit substantial zero-value sparsity. However, frequent zero values do not necessarily indicate missing information or irrelevant features. The presence or absence of nonzero values may itself carry predictive information.

**Modeling Decision:** Retain all 93 predictive features for initial modeling. Avoid treating zero values as missing data or removing zero-heavy features without evidence from model evaluation.

In [6]:
print("Duplicate train IDs:", train["id"].duplicated().sum())
print("Duplicate test IDs:", test["id"].duplicated().sum())

print(
    "Duplicate train feature rows:",
    train[feature_columns].duplicated().sum(),
)

print(
    "Duplicate test feature rows:",
    test[feature_columns].duplicated().sum(),
)

Duplicate train IDs: 0
Duplicate test IDs: 0
Duplicate train feature rows: 0
Duplicate test feature rows: 0


### Duplicate Record Analysis

**Objective:** Check for duplicate product identifiers and identical feature vectors in the training and test datasets.

**Methodology:** Count duplicated identifiers and duplicated rows across all 93 predictive features, excluding the identifier and target columns from feature-level duplicate detection.

**Findings:**
- No duplicate identifiers were found within the training or test dataset.
- No duplicate feature vectors were detected within either dataset.

**Interpretation:** Each dataset contains unique product identifiers and no repeated feature vectors. This reduces concerns about exact within-dataset duplicates influencing the analysis.

**Modeling Decision:** No duplicate removal is required. All observations will be retained for subsequent analysis and modeling.

In [7]:
train_zero_pct = train[feature_columns].eq(0).mean() * 100
test_zero_pct = test[feature_columns].eq(0).mean() * 100

zero_comparison = pd.DataFrame({
    "train_zero_pct": train_zero_pct,
    "test_zero_pct": test_zero_pct,
})

zero_comparison["abs_diff_pp"] = (
    zero_comparison["train_zero_pct"]
    - zero_comparison["test_zero_pct"]
).abs()

print(
    "Median absolute difference:",
    round(zero_comparison["abs_diff_pp"].median(), 2),
)

print(
    "Maximum absolute difference:",
    round(zero_comparison["abs_diff_pp"].max(), 2),
)

display(
    zero_comparison.nlargest(5, "abs_diff_pp").round(2)
)

Median absolute difference: 0.14
Maximum absolute difference: 0.64


,train_zero_pct,test_zero_pct,abs_diff_pp
feat_70,65.03,64.39,0.64
feat_11,72.79,72.30,0.49
feat_38,73.99,73.59,0.40
feat_32,67.81,67.41,0.40
feat_83,86.73,86.33,0.40


### Train–Test Zero-Rate Comparison

**Objective:** Assess whether the prevalence of zero values differs between the training and test datasets.

**Methodology:** Calculate the percentage of zero values for each feature in both datasets and compare the absolute differences in percentage points.

**Findings:**
- The median absolute difference across the 93 features is 0.14 percentage points.
- The largest difference is 0.64 percentage points, observed in `feat_70`.
- The five largest differences are all below one percentage point.

**Interpretation:** Zero-value frequencies appear broadly consistent between the training and test datasets. However, similar zero rates do not establish that the full feature distributions are identical.

**Modeling Decision:** No preprocessing changes are justified by the zero-rate comparison alone. Examine nonzero feature values before drawing broader conclusions about distributional similarity.

In [8]:
train_nonzero = train[feature_columns].replace(0, float("nan"))
test_nonzero = test[feature_columns].replace(0, float("nan"))

nonzero_comparison = pd.DataFrame({
    "train_mean": train_nonzero.mean(),
    "test_mean": test_nonzero.mean(),
})

nonzero_comparison["relative_diff_pct"] = (
    (nonzero_comparison["train_mean"] - nonzero_comparison["test_mean"])
    .abs()
    .div(nonzero_comparison["train_mean"])
    * 100
)

print(
    "Median relative difference (%):",
    round(nonzero_comparison["relative_diff_pct"].median(), 2),
)

print(
    "Maximum relative difference (%):",
    round(nonzero_comparison["relative_diff_pct"].max(), 2),
)

display(nonzero_comparison.nlargest(5, "relative_diff_pct").round(2))

Median relative difference (%): 0.96
Maximum relative difference (%): 6.58


,train_mean,test_mean,relative_diff_pct
feat_31,2.03,2.16,6.58
feat_68,2.78,2.65,4.98
feat_93,2.08,2.18,4.42
feat_61,2.56,2.65,3.58
feat_84,3.08,3.18,3.40


### Train–Test Nonzero Mean Comparison

**Objective:** Assess differences in nonzero feature values between the training and test datasets.

**Methodology:** Exclude zero values from the mean calculations and compare feature-wise nonzero means using absolute relative differences, with training means as the reference.

**Findings:**
- The median relative difference across the features is 0.96%.
- The largest relative difference is 6.58%, observed for `feat_31`.
- Most features exhibit relatively small differences in their nonzero means.

**Interpretation:** The results suggest broadly similar nonzero feature means across the training and test datasets. However, comparing means alone cannot establish equivalence of the full feature distributions.

**Modeling Decision:** No distribution-adjustment preprocessing is introduced based on these descriptive comparisons. All 93 predictive features will be retained for initial modeling.

In [9]:
from sklearn.model_selection import train_test_split

X = train[feature_columns]
y = train["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

class_proportions = pd.DataFrame({
    "original": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "validation": y_valid.value_counts(normalize=True),
}).sort_index() * 100

display(class_proportions.round(2))

Training shape: (49502, 93)
Validation shape: (12376, 93)


,original,train,validation
target,,,
Class_1,3.12,3.12,3.12
Class_2,26.05,26.06,26.05
Class_3,12.94,12.93,12.94
Class_4,4.35,4.35,4.35
Class_5,4.43,4.43,4.43
Class_6,22.84,22.84,22.84
Class_7,4.59,4.59,4.59
Class_8,13.68,13.68,13.68
Class_9,8.01,8.01,8.01


### Stratified Train–Validation Split

**Objective:** Establish a reproducible validation strategy while preserving the distribution of the nine target classes.

**Methodology:** Apply an 80/20 stratified train–validation split with a fixed random seed of 42.

**Findings:**
- Training set: 49,502 observations.
- Validation set: 12,376 observations.
- All 93 predictive features were retained.
- Class proportions remained nearly identical across the original, training, and validation datasets.

**Modeling Decision:** Use this fixed split for initial model comparisons. Evaluate predictive probabilities using multiclass log loss, the competition's official metric.

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss

dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(X_train, y_train)

valid_proba = dummy_model.predict_proba(X_valid)

dummy_log_loss = log_loss(
    y_valid,
    valid_proba,
    labels=dummy_model.classes_,
)

print("Dummy Validation Log Loss:", round(dummy_log_loss, 4))
print("Probability shape:", valid_proba.shape)
print("First row probability sum:", round(valid_proba[0].sum(), 6))

Dummy Validation Log Loss: 1.9505
Probability shape: (12376, 9)
First row probability sum: 1.0


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

logistic_model.fit(X_train, y_train)

logistic_valid_proba = logistic_model.predict_proba(X_valid)

logistic_valid_loss = log_loss(
    y_valid,
    logistic_valid_proba,
    labels=logistic_model.classes_,
)

print(f"Dummy Validation Log Loss: {dummy_log_loss:.4f}")
print(f"Logistic Validation Log Loss: {logistic_valid_loss:.4f}")
print(f"Probability shape: {logistic_valid_proba.shape}")

Dummy Validation Log Loss: 1.9505
Logistic Validation Log Loss: 0.6311
Probability shape: (12376, 9)


In [12]:
logistic_train_proba = logistic_model.predict_proba(X_train)

logistic_train_loss = log_loss(
    y_train,
    logistic_train_proba,
    labels=logistic_model.classes_,
)

generalization_gap = logistic_valid_loss - logistic_train_loss

print(f"Training Log Loss: {logistic_train_loss:.4f}")
print(f"Validation Log Loss: {logistic_valid_loss:.4f}")
print(f"Generalization Gap: {generalization_gap:.4f}")

Training Log Loss: 0.6228
Validation Log Loss: 0.6311
Generalization Gap: 0.0083


### Logistic Regression Baseline — Performance and Generalization

**Objective:** Evaluate the predictive performance of Logistic Regression and assess its generalization from training to validation data.

**Methodology:** Train a Logistic Regression classifier using a StandardScaler pipeline and calculate multiclass log loss on both training and validation sets.

**Findings:**
- Dummy classifier validation log loss: **1.9505**
- Logistic Regression training log loss: **0.6228**
- Logistic Regression validation log loss: **0.6311**
- Generalization gap (validation − training): **0.0083**

**Interpretation:** Logistic Regression substantially outperforms the class-prior baseline, indicating that the predictive features contain useful information about product categories. The small training-validation gap suggests limited overfitting on the current split. However, this result does not rule out underfitting or establish that a linear model is optimal.

**Modeling Decision:** Retain Logistic Regression as the first learned baseline. Next, evaluate a nonlinear tree-based model using the same validation split and multiclass log loss metric to investigate potential improvements.

In [14]:
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_valid_encoded = label_encoder.transform(y_valid)

lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    metric="multi_logloss",
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

lgbm_model.fit(
    X_train,
    y_train_encoded,
    eval_X=X_valid,
    eval_y=y_valid_encoded,
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

lgbm_train_proba = lgbm_model.predict_proba(X_train)
lgbm_valid_proba = lgbm_model.predict_proba(X_valid)

lgbm_train_loss = log_loss(
    y_train_encoded,
    lgbm_train_proba,
    labels=lgbm_model.classes_,
)

lgbm_valid_loss = log_loss(
    y_valid_encoded,
    lgbm_valid_proba,
    labels=lgbm_model.classes_,
)

print(f"Best iteration: {lgbm_model.best_iteration_}")
print(f"Training Log Loss: {lgbm_train_loss:.4f}")
print(f"Validation Log Loss: {lgbm_valid_loss:.4f}")
print(f"Generalization Gap: {lgbm_valid_loss - lgbm_train_loss:.4f}")
print(f"Logistic Validation Log Loss: {logistic_valid_loss:.4f}")

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[600]	valid_0's multi_logloss: 0.475721
Best iteration: 600
Training Log Loss: 0.2974
Validation Log Loss: 0.4757
Generalization Gap: 0.1783
Logistic Validation Log Loss: 0.6311


In [15]:
from sklearn.base import clone

lgbm_extended = clone(lgbm_model)
lgbm_extended.set_params(n_estimators=1500)

lgbm_extended.fit(
    X_train,
    y_train_encoded,
    eval_X=X_valid,
    eval_y=y_valid_encoded,
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

extended_train_proba = lgbm_extended.predict_proba(X_train)
extended_valid_proba = lgbm_extended.predict_proba(X_valid)

extended_train_loss = log_loss(
    y_train_encoded,
    extended_train_proba,
    labels=lgbm_extended.classes_,
)

extended_valid_loss = log_loss(
    y_valid_encoded,
    extended_valid_proba,
    labels=lgbm_extended.classes_,
)

print(f"Best iteration: {lgbm_extended.best_iteration_}")
print(f"Training Log Loss: {extended_train_loss:.4f}")
print(f"Validation Log Loss: {extended_valid_loss:.4f}")
print(f"Generalization Gap: {extended_valid_loss - extended_train_loss:.4f}")
print(f"Previous Validation Log Loss: {lgbm_valid_loss:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[995]	valid_0's multi_logloss: 0.468676
Best iteration: 995
Training Log Loss: 0.2256
Validation Log Loss: 0.4687
Generalization Gap: 0.2430
Previous Validation Log Loss: 0.4757


In [16]:
lgbm_reduced = clone(lgbm_extended)
lgbm_reduced.set_params(num_leaves=7)

lgbm_reduced.fit(
    X_train,
    y_train_encoded,
    eval_X=X_valid,
    eval_y=y_valid_encoded,
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

reduced_train_proba = lgbm_reduced.predict_proba(X_train)
reduced_valid_proba = lgbm_reduced.predict_proba(X_valid)

reduced_train_loss = log_loss(
    y_train_encoded,
    reduced_train_proba,
    labels=lgbm_reduced.classes_,
)

reduced_valid_loss = log_loss(
    y_valid_encoded,
    reduced_valid_proba,
    labels=lgbm_reduced.classes_,
)

print(f"Best iteration: {lgbm_reduced.best_iteration_}")
print(f"Training Log Loss: {reduced_train_loss:.4f}")
print(f"Validation Log Loss: {reduced_valid_loss:.4f}")
print(f"Generalization Gap: {reduced_valid_loss - reduced_train_loss:.4f}")
print(f"Previous Validation Log Loss: {extended_valid_loss:.4f}")

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1496]	valid_0's multi_logloss: 0.481629
Best iteration: 1496
Training Log Loss: 0.3083
Validation Log Loss: 0.4816
Generalization Gap: 0.1733
Previous Validation Log Loss: 0.4687


In [17]:
lgbm_reduced_extended = clone(lgbm_reduced)
lgbm_reduced_extended.set_params(n_estimators=3000)

lgbm_reduced_extended.fit(
    X_train,
    y_train_encoded,
    eval_X=X_valid,
    eval_y=y_valid_encoded,
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

reduced_extended_train_proba = lgbm_reduced_extended.predict_proba(X_train)
reduced_extended_valid_proba = lgbm_reduced_extended.predict_proba(X_valid)

reduced_extended_train_loss = log_loss(
    y_train_encoded,
    reduced_extended_train_proba,
    labels=lgbm_reduced_extended.classes_,
)

reduced_extended_valid_loss = log_loss(
    y_valid_encoded,
    reduced_extended_valid_proba,
    labels=lgbm_reduced_extended.classes_,
)

print(f"Best iteration: {lgbm_reduced_extended.best_iteration_}")
print(f"Training Log Loss: {reduced_extended_train_loss:.4f}")
print(f"Validation Log Loss: {reduced_extended_valid_loss:.4f}")
print(
    f"Generalization Gap: "
    f"{reduced_extended_valid_loss - reduced_extended_train_loss:.4f}"
)
print(f"Best previous Validation Log Loss: {extended_valid_loss:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1775]	valid_0's multi_logloss: 0.479202
Best iteration: 1775
Training Log Loss: 0.2850
Validation Log Loss: 0.4792
Generalization Gap: 0.1942
Best previous Validation Log Loss: 0.4687


### LightGBM Complexity and Boosting Experiments

**Objective:** Evaluate the effects of boosting duration and tree complexity on multiclass classification performance.

**Methodology:** Compare LightGBM configurations using 15 and 7 leaves per tree, with increased boosting limits and early stopping based on validation log loss.

**Findings:**
- Increasing the boosting limit for the 15-leaf model improved validation log loss from 0.4757 to 0.4687, with the best iteration at 995.
- Reducing tree complexity to 7 leaves produced a smaller generalization gap but higher validation loss.
- Extending the 7-leaf model to 3,000 possible iterations improved validation log loss to 0.4792, with the best iteration at 1,775.
- The 15-leaf model achieved the lowest validation log loss among the tested configurations.

**Interpretation:** Lower tree complexity reduced the training-validation gap but did not improve predictive performance. Additional boosting helped both configurations, although the 15-leaf model remained superior on the selected validation split.

**Modeling Decision:** Retain the 15-leaf LightGBM configuration with 995 boosting iterations as the leading candidate. Assess performance stability through stratified cross-validation before final model selection.

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

cv_model = clone(lgbm_extended)
cv_model.set_params(n_estimators=lgbm_extended.best_iteration_)

cv_results = cross_validate(
    cv_model,
    X_train,
    y_train_encoded,
    cv=cv,
    scoring="neg_log_loss",
    return_train_score=True,
    n_jobs=1,
)

cv_train_losses = -cv_results["train_score"]
cv_valid_losses = -cv_results["test_score"]

print("CV Training Log Loss:", cv_train_losses.round(4))
print("CV Validation Log Loss:", cv_valid_losses.round(4))
print(f"Mean CV Validation Log Loss: {cv_valid_losses.mean():.4f}")
print(f"CV Standard Deviation: {cv_valid_losses.std():.4f}")

CV Training Log Loss: [0.1782 0.1826 0.1801]
CV Validation Log Loss: [0.5151 0.5041 0.508 ]
Mean CV Validation Log Loss: 0.5091
CV Standard Deviation: 0.0045


In [19]:
logistic_cv_results = cross_validate(
    clone(logistic_model),
    X_train,
    y_train,
    cv=cv,
    scoring="neg_log_loss",
    return_train_score=True,
    n_jobs=1,
)

logistic_cv_train_losses = -logistic_cv_results["train_score"]
logistic_cv_valid_losses = -logistic_cv_results["test_score"]

print("Logistic CV Training Loss:", logistic_cv_train_losses.round(4))
print("Logistic CV Validation Loss:", logistic_cv_valid_losses.round(4))

print(f"Mean Logistic CV Loss: {logistic_cv_valid_losses.mean():.4f}")
print(f"Logistic CV Standard Deviation: {logistic_cv_valid_losses.std():.4f}")

print(f"Mean LightGBM CV Loss: {cv_valid_losses.mean():.4f}")

Logistic CV Training Loss: [0.6115 0.6222 0.6208]
Logistic CV Validation Loss: [0.6614 0.6412 0.6411]
Mean Logistic CV Loss: 0.6479
Logistic CV Standard Deviation: 0.0095
Mean LightGBM CV Loss: 0.5091


### Cross-Validation Model Comparison

**Objective:** Compare the generalization performance and stability of Logistic Regression and LightGBM using the same cross-validation strategy.

**Methodology:** Evaluate both models using three-fold Stratified Cross-Validation on the training partition. Report training and validation multiclass log loss for each model.

**Findings:**
- Logistic Regression achieved a mean validation log loss of **0.6479 ± 0.0095**.
- LightGBM achieved a mean validation log loss of **0.5091 ± 0.0045**.
- LightGBM reduced mean validation log loss by approximately **21.4%** compared with Logistic Regression.
- LightGBM outperformed Logistic Regression across all three folds.
- LightGBM exhibited a substantially larger training-validation gap, indicating greater overfitting.

**Interpretation:** LightGBM demonstrated superior predictive performance despite its larger generalization gap. The relatively low variation across folds provides encouraging evidence of stability, although the limited number of folds and prior holdout-based hyperparameter selection constrain the independence of this assessment.

**Modeling Decision:** Select LightGBM with `num_leaves=15` and `n_estimators=995` as the leading configuration for final training. Retain Logistic Regression as the reference baseline.

In [20]:
X_full = train[feature_columns]
y_full = train["target"]

y_full_encoded = label_encoder.transform(y_full)

final_model = clone(lgbm_extended)
final_model.set_params(n_estimators=995)

final_model.fit(X_full, y_full_encoded)

print(f"Training samples: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Trained boosting iterations: {final_model.n_estimators_}")
print(f"Classes: {label_encoder.classes_.tolist()}")

Training samples: 61878
Number of features: 93
Trained boosting iterations: 995
Classes: ['Class_1', 'Class_2', 'Class_3', 'Class_4', 'Class_5', 'Class_6', 'Class_7', 'Class_8', 'Class_9']


In [21]:
import joblib

model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

artifact_path = model_dir / "otto_lightgbm.joblib"

model_artifact = {
    "model": final_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns,
}

joblib.dump(model_artifact, artifact_path)

print(f"Artifact path: {artifact_path}")
print(f"File exists: {artifact_path.is_file()}")
print(f"File size (MB): {artifact_path.stat().st_size / (1024 ** 2):.2f}")

Artifact path: /Users/ebrietas/otto-group-product-classification/models/otto_lightgbm.joblib
File exists: True
File size (MB): 15.23


In [22]:
import numpy as np

loaded_artifact = joblib.load(artifact_path)

loaded_model = loaded_artifact["model"]
loaded_encoder = loaded_artifact["label_encoder"]
loaded_features = loaded_artifact["feature_columns"]

inference_sample = test[loaded_features].head(10)

original_proba = final_model.predict_proba(inference_sample)
loaded_proba = loaded_model.predict_proba(inference_sample)

print("Predictions match:", np.allclose(original_proba, loaded_proba))
print("Maximum probability difference:", np.max(np.abs(original_proba - loaded_proba)))
print("Feature order preserved:", loaded_features == feature_columns)
print("Class mapping preserved:", np.array_equal(
    loaded_encoder.classes_,
    label_encoder.classes_,
))
print("Prediction shape:", loaded_proba.shape)
print("First sample probability sum:", loaded_proba[0].sum())

Predictions match: True
Maximum probability difference: 0.0
Feature order preserved: True
Class mapping preserved: True
Prediction shape: (10, 9)
First sample probability sum: 0.9999999999999999


In [23]:
test_proba = loaded_model.predict_proba(test[loaded_features])

class_names = loaded_encoder.inverse_transform(
    loaded_model.classes_
)

submission = pd.DataFrame(
    test_proba,
    columns=class_names,
)

submission.insert(0, "id", test["id"].to_numpy())

print("Submission shape:", submission.shape)
print("Columns match:", submission.columns.tolist() == sample_submission.columns.tolist())
print("IDs match:", submission["id"].equals(sample_submission["id"]))
print("All probabilities finite:", np.isfinite(test_proba).all())
print("Probabilities in [0, 1]:", ((test_proba >= 0) & (test_proba <= 1)).all())
print("Row sums equal 1:", np.allclose(test_proba.sum(axis=1), 1.0))

display(submission.head())

Submission shape: (144368, 10)
Columns match: True
IDs match: True
All probabilities finite: True
Probabilities in [0, 1]: True
Row sums equal 1: True


,id,Class_1,Class_2,Class_3,Class_4,Class_5,Class_6,Class_7,Class_8,Class_9
0,1,0.000105,0.085653,0.195785,7.142917e-01,1.886535e-09,0.000069,0.003866,0.000222,8.150893e-06
1,2,0.000041,0.001632,0.000151,6.768305e-05,1.786095e-07,0.810113,0.000569,0.187327,9.855028e-05
2,3,0.000002,0.000003,0.000002,6.302343e-07,3.363664e-10,0.999900,0.000002,0.000091,1.134494e-07
3,4,0.000009,0.698296,0.299638,1.815572e-03,5.659755e-11,0.000009,0.000015,0.000052,1.654563e-04
4,5,0.116207,0.000052,0.000232,3.363491e-05,2.082315e-08,0.002473,0.007741,0.045699,8.275637e-01


In [24]:
submission_dir = project_root / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

submission_path = submission_dir / "otto_lightgbm_submission.csv"

submission.to_csv(submission_path, index=False)

print(f"Submission path: {submission_path}")
print(f"File exists: {submission_path.is_file()}")
print(f"File size (MB): {submission_path.stat().st_size / (1024 ** 2):.2f}")

Submission path: /Users/ebrietas/otto-group-product-classification/submissions/otto_lightgbm_submission.csv
File exists: True
File size (MB): 27.47


### External Evaluation — Kaggle Submission

**Objective:** Evaluate the final LightGBM model on the competition's hidden test labels and compare its performance with internal validation results.

**Methodology:** Train the selected LightGBM configuration (`num_leaves=15`, `n_estimators=995`) on all 61,878 labeled observations. Generate probability predictions for 144,368 test observations and submit the resulting CSV file to Kaggle.

**Findings:**
- Holdout validation log loss: **0.4687**
- Mean three-fold cross-validation log loss: **0.5091**
- Kaggle Public Score: **0.46207**
- Kaggle Private Score: **0.46128**
- The submission was successfully evaluated after the competition deadline.

**Interpretation:** The external test results are consistent with the strong holdout performance. The small difference between public and private scores is encouraging, although it does not eliminate concerns about overfitting or guarantee performance on future datasets. Cross-validation scores are not directly comparable because those models were trained on smaller subsets.

**Modeling Decision:** Retain the final LightGBM model and its saved artifact. No additional hyperparameter tuning is justified solely by the leaderboard results.